# Practical P4: Chat Completion & Multi-turn Conversation
**Learning Outcome**: Build a working console chatbot with conversation memory.

## Part 1: Conversation History Flowchart
Unlike traditional rule-based programs, LLMs do not keep state natively. They are stateless REST endpoints.
To simulate memory, our client must aggregate the entire conversation transcript and send it as context on every new query.

```mermaid
sequenceDiagram
  participant User
  participant Client as Python History Array
  participant LLM as AI API Endpoint
  User->>Client: Send Message 1
  Client->>LLM: Post: [Message 1]
  LLM-->>Client: Return: [Response 1]
  User->>Client: Send Message 2
  Client->>LLM: Post: [Message 1 + Response 1 + Message 2]
  LLM-->>Client: Return: [Response 2]
```


## Part 2: Building Multi-turn Conversations in Code
Let's initialize our list history and mock client.


In [ ]:
# Mock Client Setup
class MockMessages:
    def create(self, **kwargs):
        msgs = kwargs.get('messages', [])
        # Count assistant responses in messages list to verify memory presence
        ass_count = sum(1 for m in msgs if m['role'] == 'assistant')
        prompt = msgs[-1]['content']
        reply = f"[Response to '{prompt}']. Conversational history count = {ass_count}."
        return MockResponse(reply)

client = MockLLMClient(api_key='sk-mock')
client.messages = MockMessages() # Override with historical behavior mock


In [ ]:
conversation_history = []

# 1. Turn 1
user_prompt = 'My name is Hasmukh.'
conversation_history.append({'role': 'user', 'content': user_prompt})

r1 = client.messages.create(model='gpt-4o', messages=conversation_history)
conversation_history.append({'role': 'assistant', 'content': r1.content})

# 2. Turn 2
user_prompt_2 = 'What is my name?'
conversation_history.append({'role': 'user', 'content': user_prompt_2})

r2 = client.messages.create(model='gpt-4o', messages=conversation_history)
print('History Sent:', conversation_history[:-1])
print('Final AI Answer:', r2.content)


## Hands-On Exercise
**Task**: Write an interactive loop `run_console_chat()` that prompts the user keyboard input in a loop.
Append user statements and model replies to a history list. If user types `'exit'`, terminate the loop.


In [ ]:
# TODO: Complete run_console_chat using a predefined mock list for classroom test run
def run_console_chat(mock_user_statements):
    history = []
    print('AI Console Assistant initialized. (Type exit to quit)')
    
    for user_input in mock_user_statements:
        if user_input.strip().lower() == 'exit':
            print('Goodbye!')
            break
            
        print(f'\n[User]: {user_input}')
        history.append({'role': 'user', 'content': user_input})
        
        # Query API
        r = client.messages.create(model='gpt-4o-mini', messages=history)
        ans = r.content
        print(f'[AI]: {ans}')
        history.append({'role': 'assistant', 'content': ans})

run_console_chat(['Hello AI!', 'Tell me a joke.', 'exit'])
